# Import Libraries

In [44]:
import gurobipy as gp
from gurobipy import GRB
import openpyxl as opx
import gc
import DantzigWolfe_Code as dw

# Problem 1

In [46]:
## initialize class instance
problem = dw.ProductionProblem()

## solve column generation problem
history, final_master, prod_patterns = problem.column_generation()
print("\nObjective Value History:", history)

## print lambda weighted patterns
problem.print_lambda_weighted_patterns(final_master, prod_patterns)

## solve full LP
print("\nNow solve full model for comparison")
full_model = problem.full_lp()
full_model.Params.OutputFlag = 0
full_model.optimize()
print(f"Full LP Objective Value: {full_model.objVal}")


--- Iteration 1 ---
Master Problem Objective Value: 10010000.0
Dual Variables (Demand, Fixed Cost, Convexity): [10000.     0. 10000.]
Auxiliary Problem Objective Value (Max Reduced Cost): -10729959.165504826
New Pattern Added: [12.441093308199813, 7.236842105263158, 7.989347536617844, 151.41376060320465, 195.39473684210552, 251.66444740346208, 108.85956644674835, 123.02631578947344, 99.86684420772303, 38.31291234684249, 36.18421052631579, 39.94673768308922]

--- Iteration 2 ---
Master Problem Objective Value: 314469.7369953684
Dual Variables (Demand, Fixed Cost, Convexity): [  10000.         -387821.21052019   10000.        ]
Auxiliary Problem Objective Value (Max Reduced Cost): -2411175.9431466833
New Pattern Added: [6.208842897460021, 7.236842105263158, 7.989347536617844, 145.7666980244592, 195.39473684210552, 251.66444740346208, 108.65475070555034, 123.02631578947344, 99.86684420772303, 49.81185324553139, 36.18421052631579, 39.94673768308922]

--- Iteration 3 ---
Master Problem Obj

# Problem 3

### Sets and Parameters

In [5]:
## sets
V = [] ## set of cities
A = [] ## arcs between cities
P = [1,2] ## set of planes

## parameters
d = {} ## distances from any city to another
Q = 500 ## upper limit on plane 1

In [6]:
## read in station names
xl_file = opx.load_workbook('distances.xlsx')
row = 2
while xl_file["distances"].cell(row=row, column=1).value:
    V.append(xl_file["distances"].cell(row=row, column=1).value)
    row+=1

In [7]:
## function to read in an arc matrix and return values to dictionary
def read_values(xl_obj, sheet_name, nodes, dict, start_row, start_col):
    for i, origin_st in enumerate(nodes):
        for j, destin_st in enumerate(nodes):
            if xl_obj[sheet_name].cell(row=start_row+i, column=start_col+j).value:
                dict[(origin_st, destin_st)] = xl_obj[sheet_name].cell(row=start_row+i, column=start_col+j).value
    return dict

In [8]:
## read in passenger demands and flow times
d = read_values(xl_file, "distances", V, d, 2, 2)

In [9]:
## MAKE SURE TO CLOSE THE CONNECTION !!! ##
xl_file.close()

In [10]:
## generate arcs
A = [*d.keys()]

In [11]:
n = len(V)

## Formulation -- Model and Vars

In [13]:
## construct model environment
model = gp.Model("cvrp_model")

## binary variable if plane p uses an arc
x = model.addVars(A, P, vtype=GRB.BINARY, name="x") 

## auxiliary labeling variable for subtours elimination
u = model.addVars(V, P, name="u")

Set parameter Username
Academic license - for non-commercial use only - expires 2025-08-22


## Formulation -- Constraints

In [15]:
## -- Assignment Constraints -- ##
model.addConstrs( gp.quicksum( gp.quicksum( x[i,j,p] for p in P ) for j in V if (i,j) in A ) == 1 for i in V )

model.addConstrs( gp.quicksum( gp.quicksum( x[i,j,p] for p in P ) for i in V if (i,j) in A ) == 1 for j in V )

## -- Same Plane Entering/Leaving (Flow Balance Style) -- ##
model.addConstrs( gp.quicksum( x[i,j,p] for j in V if (i,j) in A ) - \
                  gp.quicksum( x[j,i,p] for j in V if (j,i) in A) == 0 for i in V for p in P )

## -- Capacity Constraint for plane 1 -- ##
model.addConstr( gp.quicksum( d[i,j]*x[i,j,1] for (i,j) in A ) <= Q )


## -- MTZ Subtour Elimination Constraints -- ##

model.addConstrs( u[i,1] - u[j,1] + 1 <= (n-1)*(1 - x[i,j,1]) for (i,j) in A if j != 'A' )

model.addConstrs( u[i,2] - u[j,2] + 1 <= (n-2)*(1 - x[i,j,2]) for (i,j) in A if j != 'F' )



#model.addConstrs( u[i,2] - u[j,2] + 1 <= (n-1)*(1 - x[i,j,2]) for i in V if i != 'F' \
#                 for j in V if j != i and j != 'F' )

{('A', 'B'): <gurobi.Constr *Awaiting Model Update*>,
 ('A', 'C'): <gurobi.Constr *Awaiting Model Update*>,
 ('A', 'D'): <gurobi.Constr *Awaiting Model Update*>,
 ('A', 'E'): <gurobi.Constr *Awaiting Model Update*>,
 ('A', 'G'): <gurobi.Constr *Awaiting Model Update*>,
 ('A', 'H'): <gurobi.Constr *Awaiting Model Update*>,
 ('A', 'I'): <gurobi.Constr *Awaiting Model Update*>,
 ('A', 'J'): <gurobi.Constr *Awaiting Model Update*>,
 ('B', 'A'): <gurobi.Constr *Awaiting Model Update*>,
 ('B', 'C'): <gurobi.Constr *Awaiting Model Update*>,
 ('B', 'D'): <gurobi.Constr *Awaiting Model Update*>,
 ('B', 'E'): <gurobi.Constr *Awaiting Model Update*>,
 ('B', 'G'): <gurobi.Constr *Awaiting Model Update*>,
 ('B', 'H'): <gurobi.Constr *Awaiting Model Update*>,
 ('B', 'I'): <gurobi.Constr *Awaiting Model Update*>,
 ('B', 'J'): <gurobi.Constr *Awaiting Model Update*>,
 ('C', 'A'): <gurobi.Constr *Awaiting Model Update*>,
 ('C', 'B'): <gurobi.Constr *Awaiting Model Update*>,
 ('C', 'D'): <gurobi.Constr 

## Formulation -- Objective and Model Optimization

In [17]:
## minimization of cost to create routes
model.setObjective( gp.quicksum( d[i,j]*gp.quicksum(x[i,j,p] for p in P) for (i,j) in A ), ## arc costs
                   sense=GRB.MINIMIZE)

#model.setParam("OutputFlag", 0)

## solve model
model.update()
model.optimize()
for t in model.getVars():
    if t.X != 0:
        print('%s: %g ' % (t.varName, t.X)) 

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26100.2))

CPU model: Intel(R) Core(TM) Ultra 7 155H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 22 logical processors, using up to 22 threads

Optimize a model with 203 rows, 200 columns and 1296 nonzeros
Model fingerprint: 0xd9689f98
Variable types: 20 continuous, 180 integer (180 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [6e+00, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+02]
Presolve removed 1 rows and 2 columns
Presolve time: 0.02s
Presolved: 202 rows, 198 columns, 1332 nonzeros
Variable types: 18 continuous, 180 integer (180 binary)

Root relaxation: objective 9.000000e+01, 59 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0   90.00000    0   12          -   90.

In [19]:
model.objVal

94.0

## Print Optimal Values

In [21]:
for t in model.getVars():
    if t.X != 0:
        print('%s: %g ' % (t.varName, t.X)) 

x[A,C,1]: 1 
x[B,D,1]: 1 
x[C,B,1]: 1 
x[D,E,1]: 1 
x[E,A,1]: 1 
x[F,J,2]: 1 
x[G,I,2]: 1 
x[H,F,2]: 1 
x[I,H,2]: 1 
x[J,G,2]: 1 
u[B,1]: 2 
u[C,1]: 1 
u[D,1]: 3 
u[E,1]: 4 
u[G,2]: 2 
u[H,2]: 4 
u[I,2]: 3 
u[J,2]: 1 
